# ARGUS Text RPS Demo

Enter raw text, choose a retriever and thresholds, then run named-entity extraction plus ARGUS Retrieval Probability Score (RPS) scoring. The first run may download the NER or embedding model.

## Setup

In [ ]:
from pathlib import Path
import importlib
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "with_argus_eyes").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not find the With_Argus_Eyes repository root from this notebook location.")
src_path = repo_root / "src"
for path in (repo_root, src_path):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = lambda value: value
    display = print

# Reload the local inference code so notebook kernels do not keep stale versions.
import with_argus_eyes.inference.text_risk as text_risk_module
import with_argus_eyes.inference as inference_module
importlib.reload(text_risk_module)
importlib.reload(inference_module)

from with_argus_eyes.inference import (
    ArgusTextConfig,
    analyze_text,
    available_retrievers,
    highlight_entities,
    resolve_model_artifact,
)

print("Using ARGUS inference module:", text_risk_module.__file__)
print("score_entities starts at line:", text_risk_module.score_entities.__code__.co_firstlineno)
print("Available retrievers:", ", ".join(available_retrievers()))


## Environment configuration

In [ ]:
import os

# Edit these before running the analysis cells.
# Use "" for CPU/default device behavior, or values such as "0", "0,1", or "6,7" for specific GPUs.
CUDA_VISIBLE_DEVICES = ""

# Keep Hugging Face downloads in a predictable location. Set to "" to use your system default.
HF_CACHE_DIR = str(repo_root / "outputs" / "cache" / "huggingface")

if CUDA_VISIBLE_DEVICES:
    os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

if HF_CACHE_DIR:
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HF_HUB_CACHE"] = str(Path(HF_CACHE_DIR) / "hub")
    os.environ["HF_DATASETS_CACHE"] = str(Path(HF_CACHE_DIR) / "datasets")
    os.environ["TRANSFORMERS_CACHE"] = str(Path(HF_CACHE_DIR) / "transformers")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<default/CPU>"))
print("HF_HOME:", os.environ.get("HF_HOME", "<system default>"))

## User configuration

In [ ]:
rps_threshold = 0.3

config = ArgusTextConfig(
    retriever="contriever",
    language="en",
    ner_model="dslim/bert-base-NER",
    risk_threshold=rps_threshold,  # Internal name kept for compatibility; this is the minimum acceptable RPS.
    ner_threshold=0.5,
    order=800,
    k=50,
    text_mode="span",
    workspace_root=repo_root,
)

artifact = resolve_model_artifact(config)
print("Selected ARGUS model artifact:")
print(artifact)

## Text input

In [ ]:
text = """
St. Martin's Church in Zillis, Switzerland, is a Romanesque church best known
for its painted wooden ceiling panels dating from the 12th century. Neanderthals
inhabited Europe and Western and Central Asia during the Middle to Late Pleistocene.
""".strip()

print(text)

## Run analysis

In [ ]:
results = analyze_text(text, config)

if not results:
    print("No named entities were found with the current NER threshold.")
else:
    print(f"Scored {len(results)} entity mentions.")

## RPS results table

In [ ]:
columns = ["entity", "entity_type", "ner_score", "rps_score", "meets_threshold", "below_threshold", "retriever"]

if results:
    try:
        import pandas as pd
        display(pd.DataFrame(results)[columns].sort_values("rps_score", ascending=False))
    except ImportError:
        for row in sorted(results, key=lambda item: item["rps_score"], reverse=True):
            print({key: row[key] for key in columns})
else:
    print("Nothing to display.")

## Highlighted text

In [ ]:
if results:
    display(HTML("<div style='line-height:1.8; font-size:1rem'>" + highlight_entities(text, results) + "</div>"))
else:
    print("No highlighted entities.")

## Compact JSON output

Entity names and Retrieval Probability Scores only.

In [ ]:
import json

compact_results = [
    {"entity": row["entity"], "rps_score": float(row["rps_score"])}
    for row in sorted(results, key=lambda item: item["rps_score"], reverse=True)
]

print(json.dumps(compact_results, ensure_ascii=False, indent=2))
